# 1. Prepare the articles and embeddings

We use this notebook to build the shared article dataset and turn each article into an embedding.
Run it in the project's `uv` environment.

For duplicate article bodies, we keep the earliest-dated copy. If dates match, we keep the first
copy in the source file. We keep articles with missing authors or sections because we do not use
those fields for modeling. This dataset describes news coverage, not public opinion.

In [1]:
from pathlib import Path
import json, sys, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'pyproject.toml').exists())
ROOT = REPO/'final-project/code/modern-methods'
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT.parent))
from shared.corpus import CORPUS_DIR, audit_corpus, load_corpus, corpus_hash, digest, write_json
from shared import evaluation as ev
from src import topic_helpers as h, artifacts as a
OUT = ROOT/'Results/modern'
OUT.mkdir(parents=True, exist_ok=True)

In [2]:
started = time.perf_counter()
df, summary = audit_corpus(source=REPO/'data/self_driving_articles.csv')
assert len(df) == 7332 and summary['source_rows'] == 7337
assert corpus_hash(df) == 'c73293f76ba4b0a18d685e5724f75b1123a86c9012497fc324090989642a8f99'
assert df.article_id.is_unique and df.article.nunique() == len(df)
preparation_seconds = time.perf_counter() - started
write_json(OUT/'corpus_preparation.json', {'seconds': preparation_seconds, 'corpus_hash': corpus_hash(df), 'environment': a.environment()})
display(pd.Series({k: summary[k] for k in ['source_rows','rows','duplicate_bodies_dropped','publications','date_min','date_max']}))
publication_summary = df.groupby('publication').agg(articles=('article_id','size'), first_date=('date','min'), last_date=('date','max'), median_words=('measured_word_count','median')).sort_values('articles',ascending=False)
publication_summary.to_csv(OUT/'corpus_by_publication.csv')
display(publication_summary)
display(pd.Series(summary['missing'], name='Missing values'))
display(df.measured_word_count.describe(percentiles=[.25,.5,.75,.9,.99]))
# Record machine resources separately from software versions and stage timings.
import subprocess
hardware = a.environment()
if sys.platform == 'darwin':
    for label, key in [('cpu_model','machdep.cpu.brand_string'),('memory_bytes','hw.memsize'),('logical_cpus','hw.ncpu')]:
        query = subprocess.run(['/usr/sbin/sysctl','-n',key],capture_output=True,text=True)
        hardware[label] = query.stdout.strip() if query.returncode == 0 else 'unavailable in this execution environment'
write_json(OUT/'hardware.json',hardware)
display(pd.Series({key:hardware.get(key,'unavailable') for key in ['platform','cpu_model','memory_bytes','logical_cpus']}))


source_rows                       7337
rows                              7332
duplicate_bodies_dropped             5
publications                        25
date_min                    2016-01-01
date_max                    2020-03-26
dtype: object

,articles,first_date,last_date,median_words
publication,,,,
TechCrunch,1214,2016-01-04,2020-03-26,432.5
CNBC,977,2016-01-01,2020-03-20,456.0
Reuters,854,2016-01-04,2020-03-19,419.0
The Verge,830,2016-01-04,2020-03-19,504.5
Mashable,554,2016-01-01,2020-03-20,402.5
The New York Times,501,2016-01-04,2020-03-19,1014.0
Vox,500,2016-01-04,2020-02-28,678.0
Axios,360,2017-01-09,2019-12-20,325.5
The Hill,353,2016-01-14,2020-02-12,737.0


article_id                  0
X1                          0
Unnamed: 0                  0
date                        0
year                        0
month                       0
day                         0
author                   1379
title                       0
article                     0
url                         0
section                  3665
publication                 0
title_match                 0
body_hits                   0
word_count                  0
text                        0
duplicate_body              0
measured_word_count         0
word_count_difference       0
Name: Missing values, dtype: int64

count     7332.000000
mean       760.835379
std       1189.988573
min        150.000000
25%        330.000000
50%        496.000000
75%        818.250000
90%       1283.000000
99%       5579.010000
max      28048.000000
Name: measured_word_count, dtype: float64

platform        macOS-26.6.2-arm64-arm-64bit-Mach-O
cpu_model                                  Apple M5
memory_bytes                            17179869184
logical_cpus                                     10
dtype: object

## Set aside articles for human review

We reserve 100 articles before reviewing model examples or naming topics. These articles still
help fit the models, but we do not use them to choose a model or its topic names. Laura needs
the same reserved IDs for the LDA analysis.

We exclude articles from the earlier audit and selection-review examples. We also exclude audit
articles with blank review fields because they may already have been read. The saved sample file
records the eligible articles and exclusions. Blank audit fields are left blank; they do not tell
us whether a review was completed.

In [3]:
audit = pd.read_csv(CORPUS_DIR/'corpus_review.csv', dtype={'article_id':str})
exclusions = set(audit.article_id)
legacy = ROOT/'selection_review/selection_review.csv'
if legacy.exists():
    old_review = pd.read_csv(legacy)
    exclusions.update(value.strip() for group in old_review.representative_articles.dropna() for value in str(group).split('|'))
reservation = ev.reserve_review_ids(df, OUT/'review_sample.json', exclusions=exclusions)
assert not set(reservation['article_ids']) & exclusions
print(f"Reserved {len(reservation['article_ids'])} articles from {len(reservation['eligible_article_ids'])} eligible records.")
print('Audit rows with all three flags recorded:', audit[['relevance','boilerplate','malformed']].notna().all(axis=1).sum())

Reserved 100 articles from 7276 eligible records.
Audit rows with all three flags recorded: 0


## Create embeddings from the full articles

[GTE-small](https://huggingface.co/thenlper/gte-small) accepts up to 512 tokens and produces a
384-dimensional vector. To include the full article, we split it into consecutive chunks of at
most 480 content tokens, add the model's special tokens, and embed each chunk.

We combine the chunk vectors using a weighted average, where the weight is the number of tokens
in each chunk. Then we scale the article vector to length 1:

`normalize(sum(chunk_token_count * chunk_vector) / total_token_count)`

The helper uses a fixed model version, records the device, checks the vectors, and saves progress
after each article so an interrupted run can resume. Averaging may lose some detail when an article
covers several topics. The optional 200-article pilot checks that the embedding code works before
running the full dataset.

In [4]:
RUN_PILOT = False
if RUN_PILOT:
    pilot = df.sample(200, random_state=42).sort_index()
    pilot_vectors, pilot_cache = h.embed_corpus(pilot, ROOT, pilot=True)
    display(pd.read_csv(pilot_cache/'chunk_diagnostics.csv').describe())

In [5]:
# Reuse a complete, verified cache without loading the encoder again.
pointer_path = OUT/'embedding_pointer.json'
complete_cache = False
if pointer_path.exists():
    existing_pointer = json.loads(pointer_path.read_text())
    complete_cache = (ROOT/existing_pointer['folder']/'embeddings.npy').exists()
if complete_cache:
    cached_df, embeddings = h.load_inputs(ROOT)
    pointer = json.loads((OUT/'embedding_pointer.json').read_text())
    cache = ROOT/pointer['folder']
    print('Reused complete embedding matrix; no new embedding timing is claimed.')
else:
    embeddings, cache = h.embed_corpus(df, ROOT)
h.validate_embeddings(embeddings, len(df))
print('Article embedding shape:', embeddings.shape)
diagnostics = pd.read_csv(cache/'chunk_diagnostics.csv')
display(diagnostics[['chunks','tokens','largest_chunk']].describe())
diagnostics[['chunks','tokens','largest_chunk']].describe().to_csv(OUT/'embedding_summary.csv')
write_json(OUT/'embedding_execution.json', json.loads((cache/'timings.json').read_text()))
display(json.loads((cache/'timings.json').read_text()))

Reused complete embedding matrix; no new embedding timing is claimed.
Article embedding shape: (7332, 384)


,chunks,tokens,largest_chunk
count,7332.000000,7332.000000,7332.000000
mean,2.550327,991.643753,439.960857
std,3.306356,1576.283467,73.670846
min,1.000000,184.000000,184.000000
25%,1.000000,427.750000,427.750000
50%,2.000000,647.000000,480.000000
75%,3.000000,1071.000000,480.000000
max,80.000000,38083.000000,480.000000


{'first_execution': {'hardware': 'macOS-26.6.2-arm64-arm-64bit-Mach-O',
  'python': '3.13.11 (main, Jan 14 2026, 23:14:53) [Clang 21.1.4 ]',
  'versions': {'torch': '2.14.1',
   'sentence-transformers': '6.1.0',
   'transformers': '5.18.0',
   'numpy': '2.5.3'},
  'model_load_download_seconds': 0.3063348749419674,
  'embedding_seconds': 100.67693650000729,
  'reused_articles': 0,
  'device': 'mps'},
 'latest_execution': {'hardware': 'macOS-26.6.2-arm64-arm-64bit-Mach-O',
  'python': '3.13.11 (main, Jan 14 2026, 23:14:53) [Clang 21.1.4 ]',
  'versions': {'torch': '2.14.1',
   'sentence-transformers': '6.1.0',
   'transformers': '5.18.0',
   'numpy': '2.5.3'},
  'model_load_download_seconds': 0.3063348749419674,
  'embedding_seconds': 100.67693650000729,
  'reused_articles': 0,
  'device': 'mps'}}

## Next steps

Both methods use `final-project/data/corpus.csv` and its manifest, which records the article IDs
and a hash used to check that the data match. Share `Results/modern/review_sample.json` with Laura
before she reviews articles for topic naming. Then continue with notebook 02.

We record data preparation, model loading, and embedding times separately. Loading saved embeddings
is much faster than creating them, so these times should be reported separately too.